# Domain Specialization — Aggregate Routing Statistics + UMAP

Generates `domain_specialization.json` (aggregate per-domain routing statistics:
`activation_rate`, `avg_prob`, `specialization_score`, `layer_divergence`, `domain_rate`,
`top_specialists`) plus `domain_specialization_umap.json` (2D UMAP projection of the same
per-(layer, expert) activation-rate vectors, one dimension per domain).

6 domains (`code`, `math`, `biomedical`, `legal`, `creative_writing`, `conversational`), five
long (~250-320 word) passages each, concatenated into one token stream per domain. code, math,
biomedical and legal each open with a concrete problem and close with an open-ended question;
creative_writing and conversational stay descriptive.
No dedicated "baseline" passage: since none of these 6 domains is meant to be neutral/generic
text, `specialization_score` and `layer_divergence` are computed against a **synthetic
baseline** — the mean activation rate across the 6 domains themselves, per (layer, expert) —
instead of a 7th hand-authored passage.

Run on a Colab A100 GPU runtime.


In [1]:
import importlib.util
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


if importlib.util.find_spec("torch") is None:
    pip_install("torch")

pip_install("transformers>=5.0.0,<6.0.0", "accelerate", "umap-learn", "numpy", "scikit-learn")

print("Dependency installation complete.")


Dependency installation complete.


In [2]:
import json
import os
from collections import defaultdict

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "allenai/OLMoE-1B-7B-0924"
OUT_PATH = "domain_specialization.json"
UMAP_OUT_PATH = "domain_specialization_umap.json"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# No attn_implementation="eager" needed here -- this script never extracts attention
# weights, only router_logits, so the default (faster) sdpa attention is fine.
model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    output_loading_info=True,
)
model.eval()

assert not loading_info["missing_keys"], (
    f"Some model weights were NOT loaded from the checkpoint (randomly initialized "
    f"instead): {loading_info['missing_keys']}"
)
print(f"unexpected_keys (informational): {loading_info.get('unexpected_keys', [])}")

config = model.config
assert config.num_hidden_layers == 16, f"Expected 16 layers, got {config.num_hidden_layers}"
assert config.num_experts == 64, f"Expected 64 experts, got {config.num_experts}"
assert config.num_experts_per_tok == 8, f"Expected top-8 routing, got {config.num_experts_per_tok}"

num_experts = config.num_experts
top_k_experts = config.num_experts_per_tok
num_layers = config.num_hidden_layers

print(f"Loaded {MODEL_ID}: {num_layers} layers, {num_experts} experts, top-{top_k_experts} routing")


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/287k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

unexpected_keys (informational): set()
Loaded allenai/OLMoE-1B-7B-0924: 16 layers, 64 experts, top-8 routing


In [3]:
# 6 domains x 5 long, coherent, grammatical passages each (30 forward passes total).
# Loading the model once is the fixed cost and prompt length barely affects memory, so long
# passages capture far richer per-domain routing signal than short ones would.
#
# code / math / biomedical / legal each open with a concrete problem and close with an
# open-ended question about how to solve it -- the shape real domain text takes when someone
# is actually working in that domain, rather than encyclopedia prose about it.
# creative_writing and conversational stay descriptive: passage 1 of each is the original
# shipped passage, kept deliberately, and passages 2-5 are new in the same register.
#
# All 5 passages per domain are concatenated into ONE flat token list per domain downstream
# (see the sweep cell), so token_idx values in expert_token_idx are offset per prompt.
DOMAIN_PROMPTS = {
    "code": [
        "The checkout endpoint of a mid-sized online store has quietly degraded over the "
        "last six weeks, with its 95th-percentile latency climbing from a comfortable one "
        "hundred and twenty milliseconds to just over two and a half seconds. Nobody can "
        "point to a single deployment that caused it, because no single deployment did; "
        "the slowdown arrived gradually, tracking the growth of the catalog rather than "
        "any change in the code that serves the request.\n\n"
        "The handler itself looks harmless on inspection. It loads the current cart, then "
        "iterates over the cart's line items, and for each line item it fetches the "
        "associated product record, and for each product record it fetches the current "
        "inventory count from a second table so that the response can mark anything out "
        "of stock. On a cart with three items this is a handful of queries. On a saved "
        "cart with sixty items, accumulated over months of browsing, it is well over a "
        "hundred round trips to the database, each one cheap on its own and ruinous in "
        "aggregate.\n\n"
        "An earlier attempt to fix this by putting the product records behind a cache "
        "made things worse rather than better. Inventory changes constantly, so the cache "
        "was invalidated aggressively, and the resulting stampede of simultaneous "
        "refreshes on popular products produced latency spikes that were sharper and less "
        "predictable than the steady slowness they replaced. Meanwhile the database "
        "server itself reports only thirty percent CPU utilization, which is why the "
        "first three engineers to look at the problem concluded that the database was not "
        "the bottleneck.\n\n"
        "The connection pool tells a different story: it sits pinned at its maximum of twenty "
        "connections for most of the business day, and the application-side wait time for "
        "acquiring a connection now exceeds the time spent executing queries. Given all of "
        "that, how would you go about confirming where the time is actually going, and what "
        "would you change first?",
        "A payments service has begun charging a small number of customers twice for the "
        "same order, and the finance team noticed before engineering did. The duplicates "
        "are rare, roughly one in every four thousand transactions, and they cluster "
        "around periods of elevated load, which is exactly the pattern that makes them "
        "difficult to reproduce in a test environment where the service is the only thing "
        "running.\n\n"
        "The architecture is straightforward. An order service publishes a message to a "
        "queue when an order is confirmed, and a payment worker consumes that message, "
        "calls an external card processor, and records the result in its own database. "
        "The queue guarantees at-least-once delivery, which the team understood to mean "
        "that a message might occasionally be delivered more than once, and which they "
        "assumed the worker already handled because the worker checks whether a payment "
        "record exists before charging the card.\n\n"
        "That check is the problem. The worker reads the payments table, sees no existing "
        "row, calls the processor, waits for the network round trip, and only then writes "
        "the row. Two workers that pick up copies of the same message within that window "
        "both see an empty table, and both charge the card. The external processor does "
        "support an idempotency key, but the current code generates a fresh random one on "
        "every attempt, which defeats the entire mechanism.\n\n"
        "Complicating matters, the worker also retries on timeout, and a timeout does not "
        "tell it whether the charge succeeded on the processor's side or never arrived at "
        "all. The team wants a fix that is correct under concurrent delivery, correct under "
        "retry, and does not require taking a distributed lock on every payment. How would "
        "you design that, and how would you prove it works?",
        "A long-running background service written in Node.js has to be restarted every "
        "thirty-six hours or it exhausts the memory on its container and is killed by the "
        "orchestrator. The restarts are scripted now, which means the incident stopped "
        "paging anyone, which in turn means the underlying problem has gone unexamined "
        "for four months while the memory ceiling has slowly crept downward from "
        "thirty-six hours to about nineteen.\n\n"
        "The service consumes a stream of events, enriches each one with data from two "
        "internal APIs, and writes the result to a data warehouse in batches. Memory "
        "usage climbs in a smooth line rather than a staircase, and it climbs whether the "
        "event volume is high or low, which suggests something accumulating per unit of "
        "time rather than per unit of work. Heap snapshots taken an hour apart are both "
        "about four hundred megabytes larger than the last, but the diff between them is "
        "dominated by thousands of small objects with no obvious common owner.\n\n"
        "There are several plausible suspects. The service registers an event listener on "
        "a shared connection object inside a per-request function, and nothing ever "
        "removes it. It also keeps a Map keyed by correlation identifier so that "
        "late-arriving responses can be matched to their originating events, and entries "
        "are deleted on success but not on the error path. A third possibility is that "
        "the batching buffer, which is supposed to flush every five seconds, is holding "
        "references to already-written rows.\n\n"
        "Each of these would produce roughly the same shape on a memory graph, and fixing all "
        "three blindly would leave the team no wiser about which one mattered. What would you "
        "do to isolate the actual cause before changing any code?",
        "The continuous integration pipeline for a moderately large web application now "
        "fails about one run in six, and almost none of those failures represent a real "
        "defect. Engineers have learned to press the retry button reflexively, which "
        "means the suite has stopped functioning as a signal: a red build no longer tells "
        "anyone anything, and a genuine regression slipped through to production last "
        "month because three people assumed it was the usual noise.\n\n"
        "The failures are spread across roughly forty tests out of nine hundred, and they "
        "share some family resemblances. Several assert on the ordering of results "
        "returned from a query that has no explicit sort clause. Several others create a "
        "record, then immediately query for it through an endpoint that reads from a "
        "replica, and fail when replication lags by more than a few milliseconds. A third "
        "group involves timers, and fails when the machine running the suite is loaded "
        "enough that a callback scheduled for one hundred milliseconds actually fires at "
        "three hundred.\n\n"
        "There is also a category that is harder to characterize. Some tests pass in "
        "isolation and fail when the suite runs in parallel across eight workers, and the "
        "specific test that fails changes from run to run. The shared fixture database is "
        "truncated between test files but not between individual tests, and a handful of "
        "tests write rows they do not clean up, so the failure surfaces in whichever "
        "unrelated test happens to run next on the same worker.\n\n"
        "The team has a week of capacity to spend on this and wants to come out of it with a "
        "suite people trust again, not merely a lower failure rate. How would you approach "
        "it, and how would you decide what to fix first?",
        "An established application needs to split its largest table, a single orders "
        "table holding four hundred million rows, into a current-orders table and an "
        "archive, because queries against it have become slow enough that the reporting "
        "dashboard times out and the nightly backup no longer finishes before the morning "
        "traffic arrives. The migration has to happen without taking the service offline, "
        "since the business operates in every time zone and there is no maintenance "
        "window that does not cost real money.\n\n"
        "The table is written to constantly by the order service and read by six other "
        "services, three of which the team does not own and cannot change on their own "
        "schedule. Two of those services issue queries with a hard-coded table name. One "
        "of them, a finance reconciliation job, runs a long transaction once a day that "
        "reads the entire table and would hold locks for the better part of an hour if it "
        "collided with a schema change.\n\n"
        "A previous attempt at a similar migration on a smaller table went badly. The "
        "team added a column with a default value, which on their database version "
        "rewrote the whole table under an exclusive lock, and the resulting outage lasted "
        "eleven minutes before someone killed the statement. That experience left the "
        "organization cautious to the point of paralysis, and the current proposal has "
        "been sitting in review for three weeks without anyone willing to approve it.\n\n"
        "The constraint that matters most is reversibility: at every step, the team wants to "
        "be able to roll back within a minute without losing writes that landed in the "
        "meantime. How would you sequence this migration, and what would you put in place to "
        "verify at each step that it is safe to continue?",
    ],
    "math": [
        "A family is considering a mortgage of three hundred and twenty thousand on a "
        "thirty-year term, quoted at a nominal annual interest rate of six percent "
        "compounded monthly. The lender's website reports a monthly payment figure, but "
        "the family would like to understand where that number comes from rather than "
        "trusting it, partly because they are also weighing a fifteen-year term at a "
        "lower quoted rate and want to compare the two on the same footing.\n\n"
        "The underlying structure is a geometric series. Each monthly payment does two "
        "things at once: it covers the interest accrued on the outstanding balance during "
        "that month, and it retires some portion of the principal. Early in the term the "
        "interest share dominates, so the balance falls slowly; late in the term the "
        "relationship inverts and the balance falls quickly. The payment itself stays "
        "constant, which is precisely the condition that pins down its value.\n\n"
        "Formally, the present value of a stream of equal payments discounted at the "
        "monthly rate must equal the amount borrowed. With a monthly rate of one half of "
        "one percent and three hundred and sixty payments, this becomes a sum of three "
        "hundred and sixty terms, each the payment divided by one plus the monthly rate "
        "raised to the appropriate power. That sum collapses because the terms form a "
        "geometric progression with a constant ratio.\n\n"
        "Comparing the two terms fairly requires care, because the quoted rates differ "
        "and the number of payments differs, so neither the monthly payment nor the total "
        "interest alone settles the question. The shorter term has a higher monthly "
        "obligation but retires principal much faster, and the difference in total "
        "interest between the two is far larger than the difference in the quoted rates "
        "would suggest, which is a general property of amortization rather than anything "
        "specific to these numbers.\n\n"
        "The family also wants to know the total interest paid over the life of the loan, and "
        "how much of the very first payment goes to principal, since both figures shape the "
        "comparison against the shorter term. How would you set up and evaluate these "
        "calculations?",
        "A ladder eight meters long is leaning against a vertical wall, and its base is "
        "being pulled away from the wall along level ground at a steady rate of half a "
        "meter per second. At the instant when the base is six meters from the wall, the "
        "question is how fast the top of the ladder is sliding down, and whether that "
        "speed is constant or changing as the base continues to move.\n\n"
        "The relationship between the two distances is fixed by the ladder's length, "
        "which does not change. If the base is at a horizontal distance from the wall and "
        "the top is at a vertical height above the ground, then the square of the "
        "horizontal distance plus the square of the vertical height always equals "
        "sixty-four. This is a constraint that holds at every instant, not just at the "
        "moment of interest, which is exactly what makes it useful.\n\n"
        "Because the constraint holds continuously, it can be differentiated with respect "
        "to time. Doing so relates the rate of change of the horizontal distance, which "
        "is given, to the rate of change of the vertical height, which is what we want. "
        "The result is an equation in which both rates appear linearly, weighted by the "
        "current values of the two distances.\n\n"
        "Substituting the known values at the instant of interest requires first "
        "recovering the vertical height, which follows from the constraint itself: with "
        "the base at six meters and the ladder at eight, the height is the square root of "
        "sixty-four minus thirty-six. That value then appears as a coefficient in the "
        "differentiated equation, and solving for the vertical rate is a single division. "
        "The sign of the result matters and should be interpreted rather than discarded, "
        "since it records that the top is descending while the base advances.\n\n"
        "There is a second, more interesting question hiding here: as the base approaches "
        "eight meters from the wall, the computed downward speed of the top grows without "
        "bound, which cannot be physically true. How would you carry out the calculation, and "
        "how would you explain what the model is getting wrong near that limit?",
        "A factory produces a component on two lines. Line A makes sixty percent of the "
        "output and has a defect rate of two percent; line B makes the remaining forty "
        "percent and has a defect rate of five percent. A quality inspector pulls one "
        "component at random from the combined output, tests it, and finds it defective. "
        "The natural question from the floor manager is which line it most likely came "
        "from.\n\n"
        "The instinct of most people hearing this is to answer line B, since line B has "
        "the higher defect rate. That instinct is not wrong about direction but it is not "
        "an answer, because it ignores the fact that line A contributes half again as "
        "many components to the pool in the first place. The correct comparison weighs "
        "each line's defect rate against its share of production, and the two effects "
        "pull in opposite directions.\n\n"
        "The structure here is a conditional probability run backwards. We know the "
        "probability of a defect given the line, and we want the probability of the line "
        "given a defect. Getting from one to the other requires the total probability of "
        "drawing a defective component at all, which is the sum over both lines of the "
        "line's production share multiplied by its defect rate.\n\n"
        "It helps to make the numbers concrete by imagining ten thousand components "
        "rather than working in fractions. Six thousand come from line A and two percent "
        "of those, one hundred and twenty, are defective. Four thousand come from line B "
        "and five percent of those, two hundred, are defective. The inspector's defective "
        "component is therefore one of three hundred and twenty, of which two hundred "
        "came from line B, and the whole conditional probability falls out of that count "
        "without any formula being invoked at all.\n\n"
        "The manager also wants to know how the answer would shift if line B's defect rate "
        "were halved after a maintenance program, and whether there is a defect rate at which "
        "the two lines become equally likely suspects. How would you compute these, and what "
        "does the second question ask you to solve for?",
        "A laboratory has measured the same physical quantity at seven different settings "
        "and obtained seven pairs of numbers that, when plotted, fall close to but not "
        "exactly on a straight line. The instrument is known to have random error of a "
        "few percent, so no line will pass through every point, and the team wants the "
        "line that best represents the underlying relationship along with an honest "
        "statement of how well it does.\n\n"
        "Because there are seven equations and only two unknowns, the slope and the "
        "intercept, the system is overdetermined and has no exact solution. Asking for "
        "one is the wrong question. The right question is which choice of slope and "
        "intercept makes the collection of residuals, the vertical gaps between each "
        "observation and the line, as small as possible under some agreed measure of "
        "smallness.\n\n"
        "The conventional measure is the sum of the squared residuals, chosen partly "
        "because squaring removes the sign and penalizes large misses more heavily, and "
        "partly because the resulting minimization has a clean closed-form answer. "
        "Setting the partial derivatives of that sum with respect to slope and intercept "
        "to zero yields two linear equations in two unknowns, which can be solved "
        "directly.\n\n"
        "There is a warning worth stating before any of this is carried out. Least "
        "squares assumes the errors live in the measured quantity and not in the setting, "
        "that they are roughly the same size across the range, and that no single "
        "observation exerts undue influence on the answer. If the instrument's error "
        "grows with the magnitude being measured, which is common, then the largest "
        "observations dominate the sum of squares and the fitted line is pulled toward "
        "them, and a weighted fit is the appropriate correction rather than a refinement.\n\n"
        "The team also wants a measure of how much of the variation in the observations the "
        "line actually explains, and a way to detect whether a straight line is the wrong "
        "shape entirely rather than merely an imprecise one. How would you carry out the fit, "
        "and what would you examine afterwards to check that it was appropriate?",
        "A savings scheme promises to pay one thousand at the end of the first year, and "
        "thereafter an amount that shrinks by a fixed factor of four fifths every year, "
        "continuing indefinitely. A prospective buyer wants to know what this stream is "
        "worth today, given that money available now can be invested elsewhere at five "
        "percent a year, and whether the infinite duration of the promise makes the value "
        "infinite too.\n\n"
        "It does not, and the reason is worth stating carefully. Each successive payment "
        "is smaller than the last by a constant ratio, and each is also discounted more "
        "heavily because it arrives later. The two effects compound, so the terms of the "
        "sum shrink geometrically with a ratio strictly less than one, and a geometric "
        "series with such a ratio converges to a finite total no matter how many terms it "
        "has.\n\n"
        "The general condition is that a geometric series converges precisely when the "
        "absolute value of its common ratio is less than one, and in that case the sum "
        "equals the first term divided by one minus the ratio. Applying this here "
        "requires identifying the effective ratio, which combines the four fifths "
        "shrinkage with the discount factor arising from the five percent alternative "
        "return.\n\n"
        "It is worth noticing where the intuition that infinite promises are infinitely "
        "valuable actually goes wrong. It is not that the payments eventually stop, "
        "because they never do; it is that they shrink faster than the number of them "
        "grows. The same structure explains why a repeating decimal is an ordinary "
        "rational number and why a bouncing ball that never quite stops nevertheless "
        "travels a finite total distance, and the failure case is equally instructive: a "
        "series whose terms shrink but not geometrically, such as one over each "
        "successive integer, diverges despite every term going to zero.\n\n"
        "The buyer would also like to know how sensitive the answer is to the assumed return, "
        "since a change from five percent to three percent seems small but may not be, and at "
        "what shrinkage factor the whole thing would fail to converge. How would you "
        "determine the present value and answer those two follow-up questions?",
    ],
    "biomedical": [
        "A sixty-four-year-old man presented to the emergency department with four days "
        "of fever, a productive cough, and increasing breathlessness on minimal exertion. "
        "His oxygen saturation on room air was eighty-nine percent, his respiratory rate "
        "was twenty-eight breaths per minute, and he was mildly confused about the date, "
        "which his daughter reported was new for him that morning.\n\n"
        "His history includes chronic obstructive pulmonary disease, managed with an "
        "inhaled corticosteroid and a long-acting bronchodilator, and type two diabetes "
        "with a recent glycated hemoglobin of eight point four percent. He was "
        "hospitalized eleven months ago for a similar episode and was discharged after "
        "five days on oral antibiotics, though the discharge summary records no organism "
        "and no de-escalation. He has not received a pneumococcal vaccination and "
        "declined influenza vaccination this season. He lives alone, and his daughter "
        "reports that he has been eating very little for the past week.\n\n"
        "Initial investigations showed a white cell count of eighteen thousand with a "
        "neutrophil predominance, a C-reactive protein of two hundred and ten milligrams "
        "per liter, and a chest radiograph with consolidation in the right lower lobe. "
        "Blood cultures were drawn before antibiotics were started. His creatinine was "
        "elevated above his documented baseline, and his blood urea nitrogen was "
        "disproportionately high, raising the question of whether he was simply "
        "dehydrated or something more was happening. Lactate was two point eight "
        "millimoles per liter on the initial venous sample and had not been repeated four "
        "hours later.\n\n"
        "There are competing framings on the ward round. One is straightforward "
        "community-acquired pneumonia in a patient with poor respiratory reserve, "
        "treatable with a standard regimen and supplemental oxygen. Another is early "
        "sepsis with organ dysfunction, in which the confusion and the rising creatinine "
        "are the first two organs to declare themselves and the tempo of treatment "
        "matters more than its precise composition. A third is that the recent "
        "hospitalization and the undocumented prior organism raise the possibility of a "
        "resistant pathogen that a standard regimen would miss entirely.\n\n"
        "The admitting team must decide on the site of care, the empiric antimicrobial "
        "regimen, and whether the confusion reflects hypoxia, sepsis, or an unrelated "
        "process. How would you structure the differential and the initial workup, and what "
        "would change your management most?",
        "A phase three trial of a new oral agent for moderate rheumatoid arthritis "
        "reported a primary endpoint result with a p-value of zero point zero four eight, "
        "narrowly below the conventional threshold. The absolute difference in the "
        "proportion of patients reaching the response criterion was six percentage "
        "points, forty-one percent on the active drug versus thirty-five percent on "
        "placebo, in a trial that randomized just over nine hundred participants.\n\n"
        "Several features of the conduct complicate the interpretation. Nineteen percent "
        "of participants discontinued before the primary assessment at twenty-four weeks, "
        "and discontinuation was unevenly distributed, with more dropouts in the placebo "
        "arm attributed to lack of efficacy. The analysis imputed non-response for all "
        "dropouts, which is conservative in one direction but, given the imbalance, may "
        "not be conservative overall.\n\n"
        "There were also four secondary endpoints, none of which reached significance, "
        "and a pre-specified subgroup analysis suggesting a substantially larger effect "
        "in patients who had failed two or more prior therapies. That subgroup finding is "
        "being emphasized in early communications about the trial, despite comprising "
        "fewer than two hundred patients and despite the trial not being powered for "
        "subgroup comparisons. No adjustment for multiplicity was applied to any of the "
        "secondary or subgroup analyses.\n\n"
        "The safety picture is unremarkable at twenty-four weeks but the follow-up is "
        "short for a drug intended to be taken for years. Serious adverse events occurred "
        "at a similar rate in both arms, with a numerical excess of herpes zoster on the "
        "active drug that the authors describe as consistent with the known class effect. "
        "Two malignancies were reported, both in the active arm, which the investigators "
        "judged unrelated. The comparator is placebo rather than an active agent already "
        "in the formulary, so the trial establishes that the drug beats nothing, not that "
        "it beats what patients would otherwise receive.\n\n"
        "A formulary committee now has to decide whether the evidence supports adding the "
        "agent, restricting it to a defined population, or waiting. How would you appraise "
        "this trial, and which uncertainties would you regard as decisive?",
        "An intensive care unit has recorded a threefold increase over eighteen months in "
        "infections caused by carbapenem-resistant organisms, concentrated among patients "
        "with prolonged mechanical ventilation and indwelling central venous catheters. "
        "Two of the isolates in the last quarter were resistant to every agent routinely "
        "tested, leaving only combination regimens with substantial toxicity as options.\n\n"
        "Review of prescribing showed that broad-spectrum agents are frequently started "
        "empirically and infrequently narrowed once culture results return. The median "
        "duration of empiric meropenem was six days even when cultures were negative at "
        "forty-eight hours. Documentation of an indication and a planned stop date was "
        "present in fewer than a third of the charts audited.\n\n"
        "There are also structural contributors. Bed occupancy has run above ninety-five "
        "percent for most of the period, nurse-to-patient ratios have been below target, "
        "and hand hygiene audit compliance has fallen from eighty-eight percent to "
        "sixty-nine percent. Environmental sampling identified the same resistant "
        "organism on two shared pieces of equipment that move between bays, though the "
        "sampling was not systematic enough to establish it as the transmission route, "
        "and no isolates were sequenced, so clonal spread and independent selection "
        "remain indistinguishable on the evidence available.\n\n"
        "The microbiology service is under-resourced and reports final identification and "
        "susceptibility at a median of seventy-two hours, which is well past the point at "
        "which most prescribing decisions have already hardened into routine. Rapid "
        "molecular testing is available but is ordered on fewer than one admission in "
        "ten, mostly by two consultants who happen to know it exists. There is no "
        "infectious diseases input on the daily round, and the pharmacist assigned to the "
        "unit covers three other wards and attends when time permits.\n\n"
        "The unit's leadership wants an intervention that reduces resistant infections "
        "without leaving genuinely septic patients undertreated, which is the tension at the "
        "heart of every stewardship program. How would you investigate the drivers here and "
        "what would you prioritize?",
        "A fifty-two-year-old woman with type two diabetes of nine years' duration "
        "returns for review with a glycated hemoglobin of nine point one percent, up from "
        "seven point eight percent a year ago. She takes metformin at the maximum "
        "tolerated dose and a sulfonylurea, and reports that she has been skipping the "
        "sulfonylurea on days when she works late because she has had two episodes of "
        "shakiness and sweating in the afternoon that resolved after eating.\n\n"
        "Her weight has increased by six kilograms over the year, her blood pressure is "
        "one hundred and forty-eight over ninety, and her estimated glomerular filtration "
        "rate has fallen to fifty-four milliliters per minute. Urine "
        "albumin-to-creatinine ratio is elevated at eighty milligrams per gram, which is "
        "new. She has no known cardiovascular disease, but her father had a myocardial "
        "infarction at fifty-nine.\n\n"
        "She describes her diet as unchanged and her activity as much reduced since a "
        "change of job six months ago. She is reluctant to start insulin, citing a "
        "relative who she believes deteriorated after starting it, and she is concerned "
        "about further weight gain. She has not been offered any newer agent and is not "
        "aware that options beyond insulin exist at this stage. Her lipid profile has not "
        "been checked in two years and she takes no statin.\n\n"
        "Several of the findings interact in ways that make the obvious next step less "
        "obvious. The afternoon symptoms are consistent with sulfonylurea-induced "
        "hypoglycemia, which is more likely as kidney function declines, so the drug she "
        "is skipping may be the one causing the events that make her skip it. The falling "
        "filtration rate and the new albuminuria together suggest diabetic kidney disease "
        "rather than a transient dip, and they also constrain which agents can be used "
        "and at what dose. The weight gain, the blood pressure, and the family history "
        "each add cardiovascular risk that glycemic control alone will not address.\n\n"
        "The consultation has to address glycemic control, the declining kidney function, the "
        "cardiovascular risk, and her stated preferences, which are not fully compatible with "
        "one another. How would you approach adjusting her regimen, and what would you "
        "monitor?",
        "A research group developing a blood-based biomarker for early detection of a "
        "solid tumor has found that their assay gives inconsistent results between "
        "laboratories. The same set of frozen samples, run at three sites using the same "
        "commercial kit and the same written protocol, produced concentration estimates "
        "differing by up to forty percent, with the direction of the discrepancy "
        "consistent within each site.\n\n"
        "Within a single site the assay looks acceptable. Replicate measurements of the "
        "same sample on the same day agree within eight percent, and a standard curve run "
        "alongside each plate falls within the manufacturer's stated tolerances. The "
        "problem appears only when results are compared across sites, which is precisely "
        "the setting in which a diagnostic biomarker would eventually have to work.\n\n"
        "Several candidate explanations exist. The sites use different plate readers with "
        "different wavelength calibration. Sample handling differs: one site thaws at "
        "room temperature and the others in a refrigerated block, and time to processing "
        "at collection varied from twenty minutes to over two hours in the original "
        "cohort. The kit itself has been supplied in three different lots across the "
        "study period, and lot-to-lot variation in the capture antibody would produce "
        "exactly this pattern. Freeze-thaw counts were not recorded, and at least some "
        "aliquots are known to have been thawed twice.\n\n"
        "The consequences are not merely technical. In the original cohort the cases were "
        "collected at one site and most of the controls at another, so any site effect on "
        "the measurement is confounded with disease status, and a forty percent "
        "site-to-site shift is larger than the case-control difference the group is "
        "reporting. That does not mean the biomarker is worthless, but it does mean the "
        "existing data cannot distinguish a real biological signal from a handling "
        "artifact that happens to align with how the samples were collected.\n\n"
        "Before the biomarker can be taken further, the group needs to know whether the "
        "signal they have observed is real and merely imprecisely measured, or an artifact of "
        "pre-analytical variation. How would you design the studies to distinguish these "
        "possibilities?",
    ],
    "legal": [
        "A manufacturer entered into a three-year supply agreement to deliver specialized "
        "components on a monthly schedule, with liquidated damages of two percent of the "
        "monthly order value for each week of delay. Fourteen months into the term, a "
        "fire at the manufacturer's sole qualified subcontractor destroyed the tooling "
        "required for one of the three component lines, and deliveries on that line "
        "stopped entirely for nineteen weeks.\n\n"
        "The agreement contains a force majeure clause excusing performance prevented by, "
        "among other listed events, fire, act of God, and any other cause beyond the "
        "reasonable control of the affected party. It requires written notice within ten "
        "business days of the event. The manufacturer gave notice on day sixteen, "
        "explaining that the first week was spent establishing whether the subcontractor "
        "could recover and that the delay was not prejudicial because the buyer had "
        "learned of the fire from press coverage on the day it happened.\n\n"
        "The buyer disputes the excuse on two grounds. First, it argues that "
        "sole-sourcing a critical component was itself within the manufacturer's control, "
        "so the resulting inability to perform was foreseeable and avoidable rather than "
        "beyond control. Second, it argues that the notice provision is a condition "
        "precedent, and that late notice forfeits the protection regardless of prejudice. "
        "It points to a qualification clause elsewhere in the agreement requiring the "
        "manufacturer to maintain a qualified alternate source for each line, an "
        "obligation the manufacturer concedes it did not meet.\n\n"
        "The manufacturer's answer is that the alternate-source obligation was subject to "
        "the buyer's own approval of any substitute, that it submitted two candidates for "
        "qualification in the first year, and that the buyer never responded to either "
        "submission. It also notes that the liquidated damages, applied across nineteen "
        "weeks, would exceed the total contract value of the affected line by a "
        "considerable margin, which raises the separate question of whether the clause "
        "operates as a penalty rather than as a genuine pre-estimate of loss and is "
        "therefore unenforceable in whole or in part.\n\n"
        "The buyer has withheld payment on the two component lines that were delivered "
        "without issue, asserting a right of set-off against accrued liquidated damages. How "
        "would you assess the strength of each party's position, and what would you advise "
        "the manufacturer to do next?",
        "A senior sales director resigned from a software company and joined a competitor "
        "eleven weeks later, in apparent breach of a twelve-month non-competition "
        "covenant in her employment contract. The covenant is unlimited as to geography, "
        "prohibits employment in any capacity by any business that competes with any part "
        "of the employer's operations, and is supported by no consideration beyond "
        "continued employment, having been introduced three years after she was "
        "originally hired.\n\n"
        "The employer has more concrete concerns than the covenant itself. In her final "
        "six weeks she downloaded a customer list, pricing schedules for the forthcoming "
        "year, and the renewal dates of the forty accounts she managed. A separate "
        "confidentiality clause and a twelve-month non-solicitation covenant, narrower "
        "and tied specifically to customers she had dealt with in the final year, also "
        "appear in the contract.\n\n"
        "The jurisdictional picture is untidy. The contract specifies the law of the "
        "state where the employer is headquartered, which enforces reasonable restraints "
        "and permits courts to narrow overbroad ones. She lives and worked remotely in a "
        "different state, one that voids employee non-competition covenants outright as a "
        "matter of public policy and treats a contrary choice-of-law clause as "
        "unenforceable in that respect. She has already filed a declaratory judgment "
        "action in her own state, two days before the employer's counsel sent its first "
        "letter, which puts the race to the forum in her favor as well.\n\n"
        "The competitor's position complicates matters further. It has produced a written "
        "offer letter stating that she was hired for an enterprise segment the employer "
        "does not serve, and it has undertaken in writing not to assign her to any "
        "account she managed previously. Whether that undertaking is worth anything "
        "depends on facts the employer cannot yet see, but it materially weakens any "
        "argument that irreparable harm is imminent, which is the element on which "
        "preliminary relief usually turns and the one least susceptible to being cured "
        "later by damages.\n\n"
        "The employer wants an injunction and has asked how quickly one could realistically "
        "be obtained. How would you evaluate which of these covenants is likely to be "
        "enforceable, and what would you advise the employer to pursue?",
        "A retailer operating across several countries wants to consolidate customer "
        "records into a single analytics platform hosted by a cloud provider whose "
        "processing occurs outside the region where most of those customers live. The "
        "proposal has been drafted on the assumption that customer consent obtained at "
        "account creation covers the transfer, and that the provider's standard "
        "contractual terms are sufficient to satisfy the applicable data protection "
        "regime.\n\n"
        "Both assumptions are doing more work than they can bear. The consent language "
        "was drafted for marketing communications and does not mention international "
        "transfer, secondary analytical use, or the identity of any recipient. Consent as "
        "a transfer basis also carries the awkward property that it can be withdrawn at "
        "any time, which would require the retailer to be able to extract and delete an "
        "individual's data from the analytics platform on request, a capability the "
        "current design does not provide.\n\n"
        "The contractual route raises its own questions. Standard clauses shift "
        "obligations onto the importer but do not by themselves address the possibility "
        "of access by public authorities in the destination country, and the applicable "
        "case law requires an assessment of the destination's legal framework and, where "
        "necessary, supplementary technical measures. The retailer has not conducted such "
        "an assessment.\n\n"
        "There is also a data minimization question that nobody has asked: whether the "
        "analytics use case genuinely requires identifiable records at all. The stated "
        "purposes are demand forecasting, basket analysis, and store-level performance "
        "reporting, none of which obviously needs to know which individual bought what, "
        "and a pseudonymized or aggregated feed would change the compliance posture "
        "substantially rather than merely improving it at the margin.\n\n"
        "Two further points sit unaddressed in the draft. The retention schedule is "
        "inherited from the operational systems and would keep records for seven years in "
        "a platform whose purpose is analytical rather than transactional, and no record "
        "of processing activities has been prepared for the new processing, which is an "
        "obligation independent of whether the transfer itself is lawful.\n\n"
        "How would you structure a compliant approach, and in what order would you address "
        "these issues?",
        "A startup engaged an independent contractor to build the core matching engine of "
        "its product over eight months, paying a fixed monthly fee under a two-page "
        "agreement that describes the deliverables but says nothing about intellectual "
        "property ownership. The engine now underpins the entire business, and the "
        "startup is midway through a financing round in which the investor's counsel has "
        "flagged the ownership position as a condition to closing.\n\n"
        "The contractor's position is that, absent an express assignment, authorship and "
        "therefore ownership of the copyright in the code remained with him, and that the "
        "startup received at most an implied license to use what he delivered. He has "
        "offered to assign the rights for a payment he characterizes as reflecting the "
        "present value of the work rather than what he was originally paid. The startup "
        "regards this as opportunistic given that he was compensated in full and on time.\n\n"
        "The record is mixed. He worked from his own equipment on his own schedule and "
        "simultaneously served three other clients, which cuts against any argument that "
        "he was in substance an employee. On the other hand, he attended the startup's "
        "daily standups, his commits were merged through the startup's review process, "
        "and two of the startup's own engineers contributed substantially to the same "
        "files, creating a plausible argument of joint authorship in at least part of the "
        "codebase.\n\n"
        "There are also facts that cut across the ownership question entirely. The "
        "engagement letter contains a broad confidentiality clause and a clause requiring "
        "the return of all materials on termination, neither of which he complied with, "
        "and he has retained a private copy of the repository. Two of the third-party "
        "libraries he introduced are licensed on terms that require source disclosure for "
        "derivative works, an issue the investor's counsel has not yet reached but will.\n\n"
        "The financing timeline leaves roughly six weeks. How would you analyze the ownership "
        "position, and what practical options would you present to the startup?",
        "The board of a closely held company has approved a merger in which the "
        "controlling shareholder, who holds fifty-eight percent of the voting stock and "
        "also serves as chief executive, will roll over her equity into the surviving "
        "entity while the minority shareholders receive cash. The price was negotiated "
        "between the controller and a buyer she introduced, and was approved by a board "
        "on which three of five directors have ongoing business relationships with her.\n\n"
        "A minority shareholder holding nine percent has objected. He points out that no "
        "special committee of independent directors was formed, that the sole fairness "
        "opinion was obtained from a bank that had advised the company on two prior "
        "transactions and stands to earn a success fee, and that the process ran from "
        "first contact to signed agreement in five weeks with no market check of any "
        "kind.\n\n"
        "The company responds that the price represents a thirty-one percent premium to "
        "the last trade, that the minority holders are being paid in cash while the "
        "controller assumes the risk of the combined business, and that a majority of the "
        "minority shares were voted in favor after full disclosure of the controller's "
        "continuing interest. The disclosure, however, omitted that the buyer had earlier "
        "indicated willingness to consider a materially higher price for a transaction "
        "structured without the rollover.\n\n"
        "The two routes he is weighing behave very differently. An appraisal action asks "
        "only what the shares were worth and does not require proving that anyone did "
        "anything wrong, but it is procedurally unforgiving on deadlines and voting "
        "conduct, and in a company with no liquid market the valuation fight would be "
        "expensive and largely a battle of experts. A fiduciary duty claim puts the "
        "process itself in issue, and the absence of a special committee together with a "
        "disclosure defect in the minority vote would likely deny the defendants the "
        "deferential review they are relying on.\n\n"
        "He is also under time pressure of a kind that shapes the choice: the transaction "
        "is scheduled to close in three weeks, and the perfection requirements for one of "
        "these routes must be satisfied before the vote he has already cast can be "
        "undone.\n\n"
        "How would you evaluate the process and the likely standard of review, and what would "
        "you advise him to do?",
    ],
    "creative_writing": [
        "The lighthouse keeper had watched a thousand storms roll in off the grey "
        "Atlantic, but something about this one made him pause at the window with his tea "
        "going cold in his hand. The waves were climbing higher than the rocks that had "
        "stood against them for three hundred years, and for the first time in his forty "
        "seasons on the island, he found himself counting the ships he could see and "
        "hoping the count would not change by morning.\n\n"
        "Mira found the letter tucked inside a hollowed-out book on her grandmother's "
        "shelf, the paper gone soft and yellow at the folds. Her hands trembled as she "
        "unfolded it, not from cold but from the particular fear of learning something "
        "that could not be unlearned, and when she finally read the first line, she "
        "understood at once why it had been hidden rather than simply thrown away.\n\n"
        "Deep in the forest, where the canopy grew so thick that noon light arrived the "
        "color of dusk, the old paths remembered every traveler who had ever walked them. "
        "The wind moved through the high branches in long, unhurried sighs, and if you "
        "stood still long enough and let your own breathing slow to match it, you could "
        "almost believe the trees were arguing quietly among themselves about whether to "
        "let you pass.\n\n"
        "By the time the last streetlamp flickered out, the city had already begun its other "
        "life, the one that belonged to the people who swept its floors and stocked its "
        "shelves while everyone else slept. A fox slipped across the empty intersection "
        "without breaking stride, entirely unbothered by the traffic lights still cycling to "
        "no one, and somewhere above the rooftops the sky was already deciding, slowly, what "
        "color the morning would be.",
        "The train had been stopped for forty minutes in a field with no station and no "
        "announcement, and the carriage had settled into the particular silence of "
        "strangers who have privately agreed not to acknowledge one another. A woman near "
        "the door had given up on her book and was watching the wheat move in long slow "
        "waves, the way water moves when something large has passed underneath it and "
        "gone.\n\n"
        "Across the aisle a boy of perhaps nine was drawing the same horse over and over "
        "on the back of a ticket, each one a little worse than the last, each one "
        "abandoned before the legs. His mother slept with her head against the glass and "
        "her hand still curled around a paper cup that had been empty since the last "
        "town, and he was careful not to move in any way that might wake her.\n\n"
        "The conductor came through once, said nothing, and went back the way he had "
        "come, and this was somehow more alarming than any explanation would have been. "
        "Someone laughed at the far end of the carriage, a short surprised sound, and "
        "then stopped as though embarrassed to have been heard. The light outside had "
        "begun to go gold and low, and the shadows of the stalks had grown long enough to "
        "reach the ballast beside the track.\n\n"
        "When the train finally moved it did so without warning and without ceremony, a slow "
        "lurch that spilled nothing and woke no one, and the field slid away as if it had "
        "never held them at all. The boy looked up, considered the window for a moment, and "
        "went back to his horse. It was, this time, very nearly right.",
        "Every October the bakery on the corner changed its window, and the whole street "
        "took this as more reliable than any calendar. Out came the summer things, the "
        "pale fruit tarts and the lemon cakes that nobody had wanted for weeks anyway, "
        "and in their place appeared the dark, spiced, unfashionable pastries that the "
        "owner's mother had made and her mother before that, arranged with the same "
        "slight carelessness every year.\n\n"
        "The owner was seventy-one and had stopped pretending she would retire. She "
        "arrived at four, worked the dough by feel because her eyes were no longer to be "
        "trusted with a scale, and by six had produced something that no measurement "
        "could have accounted for. People who had moved away came back for it. People who "
        "had never left had stopped noticing, which she considered the higher compliment "
        "of the two.\n\n"
        "The girl who worked weekends had asked, once, to be taught, and had been given "
        "instead a broom and three months of sweeping. She had understood this "
        "eventually, though not at the time, and had stayed. Now she could tell by the "
        "sound of the mixer alone whether the batch would be right, and she had begun to "
        "be quietly frightened of the day she would have to prove it.\n\n"
        "On the first cold morning the smell reached the end of the street before the "
        "shutters were fully up, and a line formed in the dark without anyone having agreed "
        "to form one. Inside, the two of them worked without speaking, in the way of people "
        "who have divided a task so thoroughly that speech has become an interruption rather "
        "than an aid.",
        "The house had been empty for eleven years and it had not been idle. Ivy had come "
        "in under the back door and gone up the stairwell like something with a plan. A "
        "birch had rooted in the gutter and grown to the height of a man, and the roof "
        "beneath it had given way politely, in stages, so that rain now fell into the "
        "upstairs hall in a thin steady column that had worn a bowl into the floorboards.\n\n"
        "In the kitchen the table was still set for four. Somebody had meant to come "
        "back. There were cups with a residue at the bottom that had long since stopped "
        "being anything, and a newspaper folded to the crossword, three answers filled in "
        "with a confident hand and then nothing, as if the person doing it had been "
        "called away mid-thought and had never found the thread again.\n\n"
        "The garden had gone entirely feral and was the better for it. What had been a "
        "lawn was now a meadow of no particular design, thick with things that had "
        "arrived on the wind, and the apple tree that someone had once pruned into "
        "obedience had thrown out a decade of unruly growth and was carrying more fruit "
        "than it could hold. Wasps moved through the fallen ones in the grass with an air "
        "of long-established ownership.\n\n"
        "A surveyor came in the spring with a clipboard and a set of intentions, and stood "
        "for a while in the doorway without going in. He had seen forty of these. He wrote "
        "down what he had come to write down, and then, after a moment, he wrote down "
        "something else, and did not look at it again until he was back in the car.",
        "The old man walked the same three kilometers of shoreline every morning at the "
        "same hour, and had done so for so long that the dogs of the town met him at "
        "their own gates and accompanied him in shifts. He carried a canvas bag and "
        "picked up what the tide had left, sorting as he went, and by the pier he would "
        "have a bag of things worth keeping and a smaller one of things that were merely "
        "interesting.\n\n"
        "He had been a mechanic, and before that a fisherman, and before that a boy who "
        "had been told he would be a fisherman and had believed it. He did not think "
        "about any of this while walking. What occupied him instead was the surface of "
        "the water, which he read the way other people read a face, and which had been "
        "telling him for several weeks that the season was going to turn early and turn "
        "hard.\n\n"
        "There was a boy who followed him sometimes, at a distance he presumably imagined "
        "was undetectable, and who had recently begun leaving things where they would be "
        "found: a coil of good rope, a float, once a brass fitting so improbable that the "
        "old man had laughed aloud on the empty beach. He had not acknowledged any of it. "
        "He was waiting to see how long the boy would keep it up.\n\n"
        "On the morning the weather finally broke, he was already at the pier when the first "
        "squall arrived, and he stood under the overhang with his two bags and watched it "
        "come across the bay. The boy arrived a minute later, soaked, out of breath, and "
        "stood beside him without a word, and the two of them watched the water together "
        "until it passed.",
    ],
    "conversational": [
        "Hey, sorry for the late reply, my phone died on the way home and I didn't get a "
        "chance to charge it until just now. Anyway, are we still good for Saturday, or "
        "did something come up on your end? I can also do Sunday afternoon if that works "
        "better, just let me know so I can figure out the rest of my weekend around it.\n\n"
        "Honestly, I've been kind of exhausted this week, nothing serious, just one of "
        "those stretches where every day feels a little longer than it should. I think I "
        "just need a weekend where I don't have anywhere to be, maybe cook something "
        "simple, watch a movie I've already seen a dozen times, that kind of thing. How "
        "about you, anything fun happen lately, or has it been the same kind of week over "
        "there?\n\n"
        "Oh, that reminds me, did you end up trying that new place downtown? A couple of "
        "people at work were talking about it and apparently the line gets pretty long on "
        "weekends, so if we want to check it out we should probably go early or just do a "
        "weekday evening instead. I'm not picky either way, honestly whatever's easiest "
        "works for me, I just haven't had a good excuse to get out of the house in a "
        "while.\n\n"
        "Thanks again for helping me move that bookshelf last week, by the way, I really owe "
        "you one. Let me know if you ever need a hand with anything, moving, fixing something "
        "around the house, whatever, I'm around most weekends these days. Talk soon, and text "
        "me whenever about Saturday, no rush.",
        "Okay so I finally got around to booking the flights, and I have good news and "
        "slightly annoying news. Good news is the outbound is cheaper than we thought, "
        "like noticeably cheaper, so we're actually under what we budgeted even with the "
        "extra bag. Slightly annoying news is the only sensible return lands at eleven at "
        "night, which means either a very late drive back or one more night somewhere.\n\n"
        "I'm leaning towards just staying the extra night honestly. By that point we'll "
        "have been on the go for a week and I know myself, I'll be useless in the car. "
        "There's a couple of cheap places near the airport that look fine, nothing "
        "special, but we'd literally be sleeping and leaving. What do you think, is that "
        "reasonable or am I overcomplicating a two hour drive?\n\n"
        "Also I have zero plan for what we actually do while we're there, which is either "
        "relaxing or irresponsible depending on your mood. I did save a few things, "
        "mostly food places and one walk that looked nice, but I figured we'd work it out "
        "day by day rather than schedule the whole thing. You're usually better at that "
        "stuff than me anyway, so feel free to take over.\n\n"
        "One more thing, do you still have that adapter from last time or did it end up in "
        "the bag that got lost? I can just buy another one, they're a few quid, I just don't "
        "want to end up with four of them like last year. Let me know about the extra night "
        "when you get a chance, no rush, the hotel prices haven't moved in days.",
        "Right, quick update on the flat because I know you've been asking. The landlord "
        "finally sent someone about the boiler, took him about twenty minutes, and "
        "apparently it was just a pressure thing the whole time. Six weeks. Six weeks of "
        "cold showers for something that took twenty minutes. I'm trying very hard to be "
        "gracious about it and not entirely succeeding, as you can probably tell.\n\n"
        "On the plus side the radiator in the back room actually works now, which it "
        "never did before, so I've basically gained a room. I've been working in there "
        "instead of the kitchen and it's a lot better, way less noise from the street and "
        "I don't have to clear my stuff off the table every time I want to eat. Small "
        "thing but it's made the whole week feel more manageable.\n\n"
        "The neighbours have been doing something upstairs that involves a lot of "
        "drilling between nine and about four, which is exactly when I'm trying to "
        "concentrate, so it's not all good. I did go up and ask and they were completely "
        "lovely about it, said another two weeks, so now I feel like I can't complain "
        "again. Classic. Headphones it is.\n\n"
        "Anyway, you should come round once it's quieter, the place actually looks like "
        "somewhere a person lives now. I've even hung things on the walls, which I know is a "
        "low bar, but eight months is eight months. Let me know what your next few weekends "
        "look like and I'll cook, properly, not the pasta thing I always do.",
        "So, small confession, I did not go to the gym once this month. Not once. I paid "
        "for it, I walked past it maybe twenty times, and I did not go in. At this point "
        "I think I'm just donating money to a building. I keep saying I'll start again on "
        "Monday and then Monday arrives and I'm suddenly extremely busy with things I "
        "invented that morning.\n\n"
        "The annoying part is I actually like it once I'm there. It's never the workout "
        "that's the problem, it's the twenty minutes beforehand where I have to find my "
        "stuff and decide it's worth it. I'm told the trick is to just go without "
        "deciding, which sounds simple and has so far not worked for me even slightly. "
        "Maybe I need to go with someone, which is where you come in, obviously.\n\n"
        "How's yours going? You seemed pretty into it last time we spoke, are you still "
        "doing the morning thing? I genuinely don't know how you manage that, I've tried "
        "getting up early and I just end up tired all day and then awake at midnight, "
        "which feels worse than doing nothing. Some people are wired for it and I've "
        "accepted I'm probably not one of them.\n\n"
        "Anyway, if you ever want company for a session, even just a walk, let me know. I'm "
        "around most evenings and I'd probably actually show up if someone was expecting me. "
        "Otherwise I'll see you at the thing on the twelfth, assuming that's still happening, "
        "and we can talk about it in person instead of me typing paragraphs at you.",
        "Hey, hope you're doing alright. I know things have been a lot lately and I "
        "didn't want to just leave it at the message I sent last week, which in hindsight "
        "was pretty short. I've been thinking about you. No pressure to reply properly or "
        "at all, honestly, I just wanted you to know I'm around and not going anywhere.\n\n"
        "For what it's worth, when my dad was ill I got about forty messages a day and "
        "answered maybe three of them, and nobody minded, and the ones I remember now are "
        "the ones that didn't need anything back. So take that however you want. If you'd "
        "rather talk about literally anything else, I've got plenty of nonsense saved up "
        "and I'm happy to just talk at you for an hour.\n\n"
        "Practical stuff, because I know that's sometimes easier than the other kind: I "
        "can drive, I can pick things up, I can be somewhere with food and not say much. "
        "Any of those, any time, including hours that would be unreasonable to ask of "
        "most people. I mean that as an actual offer and not the thing people say.\n\n"
        "Anyway. I'll stop. Just wanted to check in properly rather than leave it hanging. "
        "Take your time with everything, and if it's useful to have a fixed thing in the "
        "diary I'm free most of the month. Otherwise I'll message again in a bit and you can "
        "ignore that one too.",
    ],
}

domains = list(DOMAIN_PROMPTS.keys())
EXPECTED_PROMPTS = 5
print(f"Domains: {domains}")
for domain, prompts in DOMAIN_PROMPTS.items():
    assert len(prompts) == EXPECTED_PROMPTS, (
        f"{domain} has {len(prompts)} prompts, expected {EXPECTED_PROMPTS}"
    )
    words = [len(p.split()) for p in prompts]
    print(f"  {domain}: {len(prompts)} prompts, {sum(words)} words total, per-prompt {words}")

Domains: ['code', 'math', 'biomedical', 'legal', 'creative_writing', 'conversational']
  code: 5 prompts, 1468 words total, per-prompt [318, 289, 282, 289, 290]
  math: 5 prompts, 1762 words total, per-prompt [351, 354, 351, 347, 359]
  biomedical: 5 prompts, 1722 words total, per-prompt [372, 338, 316, 354, 342]
  legal: 5 prompts, 1785 words total, per-prompt [376, 357, 345, 326, 381]
  creative_writing: 5 prompts, 1436 words total, per-prompt [293, 284, 275, 290, 294]
  conversational: 5 prompts, 1310 words total, per-prompt [271, 274, 259, 265, 241]


In [4]:
def stats_for_prompt(prompt):
    """Returns per-layer [num_experts] top-k hit counts and summed probs, plus token count,
    plus each token's real decoded text and, per layer/expert, which (token index, routing
    score) pairs actually selected that expert in their real top-k -- the score lets callers
    rank tokens by how strongly they activated the expert, not just occurrence order."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_router_logits=True)
    router_logits = outputs.router_logits  # tuple of [seq, num_experts], one per layer
    token_ids = inputs["input_ids"][0].tolist()
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    n_tokens = len(token_ids)

    hit_counts = torch.zeros(num_layers, num_experts)
    prob_sums = torch.zeros(num_layers, num_experts)
    expert_token_idx = [[[] for _ in range(num_experts)] for _ in range(num_layers)]
    for li, layer_logits in enumerate(router_logits):
        probs = torch.softmax(layer_logits.float(), dim=-1).cpu()  # [seq, num_experts]
        topk = torch.topk(probs, k=top_k_experts, dim=-1).indices  # [seq, top_k]
        for t in range(n_tokens):
            hit_counts[li, topk[t]] += 1
            for e in topk[t].tolist():
                expert_token_idx[li][e].append((t, float(probs[t, e])))
        prob_sums[li] += probs.sum(dim=0)
    return hit_counts, prob_sums, n_tokens, token_strs, expert_token_idx


In [5]:
activation_rate = {}    # domain -> [layer][expert]  fraction of domain's tokens with expert in top-k
avg_prob = {}           # domain -> [layer][expert]  mean router softmax prob (selected or not)
token_counts = {}
prompt_counts = {}
# domain -> [token_str, ...] and domain -> [layer][expert] -> [(token_idx, score), ...] into that
# list, so the popup can show exactly which real tokens routed to a given expert/layer.
#
# Each domain now has 5 passages, and they are aggregated into ONE flat token list per domain:
# hit counts and prob sums add up across passages and are divided by the combined token count, so
# activation_rate/avg_prob keep meaning exactly what they meant with a single passage. Each
# passage's token indices are shifted by the number of tokens contributed by the passages before
# it -- the shift is taken per passage inside the loop (len(all_tokens) before extending), never
# recomputed once at the end, and the asserts below check every index lands in range.
domain_tokens = {}
expert_token_idx = {}

for domain, prompts in DOMAIN_PROMPTS.items():
    print(f"\n== domain: {domain} ==")
    total_hits = torch.zeros(num_layers, num_experts)
    total_probs = torch.zeros(num_layers, num_experts)
    all_tokens = []
    merged = [[[] for _ in range(num_experts)] for _ in range(num_layers)]

    for pi, prompt in enumerate(prompts):
        hits, probs, n_tok, token_strs, e_idx = stats_for_prompt(prompt)
        offset = len(all_tokens)  # tokens contributed by the passages before this one
        total_hits += hits
        total_probs += probs
        all_tokens.extend(token_strs)
        for li in range(num_layers):
            for e in range(num_experts):
                merged[li][e].extend((offset + t, s) for t, s in e_idx[li][e])
        print(f"  passage {pi + 1}/{len(prompts)}: {n_tok} tokens  {prompt[:56]!r}...")

    n_total = len(all_tokens)
    activation_rate[domain] = (total_hits / n_total).tolist()
    avg_prob[domain] = (total_probs / n_total).tolist()
    token_counts[domain] = n_total
    prompt_counts[domain] = len(prompts)
    domain_tokens[domain] = all_tokens
    expert_token_idx[domain] = merged

    assert len(domain_tokens[domain]) == token_counts[domain]
    highest = max((t for row in merged for cell in row for t, _ in cell), default=-1)
    assert highest < n_total, (
        f"{domain}: token_idx {highest} does not address the {n_total}-token concatenated list "
        f"-- the per-passage offset is wrong"
    )
    print(f"  total: {len(prompts)} passages, {n_total} tokens")


== domain: code ==


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  passage 1/5: 373 tokens  'The checkout endpoint of a mid-sized online store has qu'...
  passage 2/5: 341 tokens  'A payments service has begun charging a small number of '...
  passage 3/5: 347 tokens  'A long-running background service written in Node.js has'...
  passage 4/5: 324 tokens  'The continuous integration pipeline for a moderately lar'...
  passage 5/5: 328 tokens  'An established application needs to split its largest ta'...
  total: 5 passages, 1713 tokens

== domain: math ==
  passage 1/5: 404 tokens  'A family is considering a mortgage of three hundred and '...
  passage 2/5: 405 tokens  'A ladder eight meters long is leaning against a vertical'...
  passage 3/5: 396 tokens  'A factory produces a component on two lines. Line A make'...
  passage 4/5: 398 tokens  'A laboratory has measured the same physical quantity at '...
  passage 5/5: 405 tokens  'A savings scheme promises to pay one thousand at the end'...
  total: 5 passages, 2008 tokens

== domain: biomedical =

In [6]:
# Synthetic baseline: none of the 6 domains is meant to be neutral/generic text, so instead
# of a 7th hand-authored "baseline" passage, use the mean activation rate across the 6
# domains themselves, per (layer, expert), as the reference point for specialization_score
# and layer_divergence.
EPS = 1e-4
baseline_rate = [
    [sum(activation_rate[d][li][e] for d in domains) / len(domains) for e in range(num_experts)]
    for li in range(num_layers)
]

# specialization_score[domain][layer][expert] = log2((rate_domain + eps) / (rate_baseline + eps))
# vs the synthetic baseline -- positive = over-used relative to the 6-domain average.
specialization_score = {}
for domain in domains:
    rate = activation_rate[domain]
    specialization_score[domain] = [
        [
            round(
                torch.log2(torch.tensor((rate[li][e] + EPS) / (baseline_rate[li][e] + EPS))).item(),
                4,
            )
            for e in range(num_experts)
        ]
        for li in range(num_layers)
    ]

# layer_divergence[domain][layer] = total-variation distance between domain's and the
# synthetic baseline's per-expert selection distribution (each normalized to sum to 1 across
# experts via /top_k). 0 = identical routing to the 6-domain average at that layer, 1 =
# completely disjoint expert sets. Kept as supplementary context (shown in the click popup);
# the primary chart plots domain_rate directly for all 6 domains.
layer_divergence = {}
for domain in domains:
    divs = []
    for li in range(num_layers):
        dom_dist = [activation_rate[domain][li][e] / top_k_experts for e in range(num_experts)]
        base_dist = [baseline_rate[li][e] / top_k_experts for e in range(num_experts)]
        tv = 0.5 * sum(abs(a - b) for a, b in zip(dom_dist, base_dist))
        divs.append(round(tv, 5))
    layer_divergence[domain] = divs

# domain_rate[domain][layer] = mean activation_rate of that domain's top-K (=top_k_experts)
# most-used experts at that layer -- a single self-contained number per domain per layer,
# computed identically across all 6 domains so they can be plotted side by side.
domain_rate = {}
for domain in domains:
    rates = []
    for li in range(num_layers):
        top_vals = sorted(activation_rate[domain][li], reverse=True)[:top_k_experts]
        rates.append(round(sum(top_vals) / len(top_vals), 5))
    domain_rate[domain] = rates

# top experts per domain, ranked directly by real activation_rate (no baseline comparison)
top_specialists = {}
for domain in domains:
    pairs = []
    for li in range(num_layers):
        for e in range(num_experts):
            pairs.append((activation_rate[domain][li][e], li, e))
    pairs.sort(reverse=True)
    top_specialists[domain] = [
        {"layer": li, "expert": e, "activation_rate": round(rate, 4)}
        for rate, li, e in pairs[:12]
    ]


In [7]:
# expert_token_idx is the only field that scales with prompt length, and at 5 passages per domain
# it would dominate the file (OLMoE ~40 MB uncapped, DeepSeek ~30 MB). The UI never renders more
# than a few dozen tokens for one cell, so ship only the TOKEN_CAP highest-scoring pairs per
# (domain, layer, expert), re-sorted back into passage order because that is the order the popup
# renders them in. expert_token_count carries the true untruncated pair count so the popup's
# "+N more" stays honest, and it is small (one int per cell).
#
# Deliberately a NEW dict: the UMAP cell below pools the full, uncapped expert_token_idx, and
# top-TOKEN_CAP-by-score is a superset of its top-8-by-score either way.
TOKEN_CAP = 40

expert_token_idx_out = {}
expert_token_count = {}
for d in domains:
    capped_layers, count_layers = [], []
    for li in range(num_layers):
        capped_row, count_row = [], []
        for e in range(num_experts):
            pairs = expert_token_idx[d][li][e]
            count_row.append(len(pairs))
            keep = sorted(pairs, key=lambda p: -p[1])[:TOKEN_CAP]
            keep.sort(key=lambda p: p[0])
            capped_row.append([[t, round(float(s), 5)] for t, s in keep])
        capped_layers.append(capped_row)
        count_layers.append(count_row)
    expert_token_idx_out[d] = capped_layers
    expert_token_count[d] = count_layers

kept = sum(len(c) for d in domains for row in expert_token_idx_out[d] for c in row)
total = sum(n for d in domains for row in expert_token_count[d] for n in row)
print(f"expert_token_idx: keeping {kept} of {total} (token_idx, score) pairs (cap {TOKEN_CAP})")

out = {
    "domains": domains,
    "num_layers": num_layers,
    "num_experts": num_experts,
    "top_k_experts": top_k_experts,
    "token_counts": token_counts,
    "prompt_counts": prompt_counts,
    "example_prompts": DOMAIN_PROMPTS,
    "activation_rate": {d: [[round(v, 5) for v in row] for row in activation_rate[d]] for d in domains},
    "avg_prob": {d: [[round(v, 5) for v in row] for row in avg_prob[d]] for d in domains},
    "specialization_score": specialization_score,
    "layer_divergence": layer_divergence,
    "domain_rate": domain_rate,
    "domain_tokens": domain_tokens,
    "expert_token_idx": expert_token_idx_out,
    "expert_token_count": expert_token_count,
    "top_specialists": top_specialists,
}

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f)

print(f"\nWrote domain specialization data to {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e6:.1f} MB)")

expert_token_idx: keeping 228932 of 1428224 (token_idx, score) pairs (cap 40)

Wrote domain specialization data to domain_specialization.json (4.1 MB)


## UMAP: (layer, expert) activation across domains

Reuses the `activation_rate` computed above (no extra forward passes) to build one vector
per (layer, expert) pair, one dimension per domain, and projects it to 2D with cosine-metric
UMAP -- the same method used in `extract_routing_trace.ipynb`. All-zero (never-activated)
pairs are excluded from the projection and reported separately as `excluded_experts`.


In [8]:
import umap

NUM_LAYERS = num_layers
NUM_EXPERTS = num_experts

expert_vectors = np.array([
    [activation_rate[d][li][e] for d in domains]
    for li in range(NUM_LAYERS)
    for e in range(NUM_EXPERTS)
])

point_layer_ids = np.repeat(np.arange(NUM_LAYERS), NUM_EXPERTS)
point_expert_ids = np.tile(np.arange(NUM_EXPERTS), NUM_LAYERS)

# Never-activated (layer, expert) pairs are all-zero and undefined under the cosine metric --
# exclude from the projection, report separately as excluded_experts.
active_mask = expert_vectors.sum(axis=1) > 0
active_vectors = expert_vectors[active_mask]

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, metric="cosine", n_jobs=1)
active_embedding = reducer.fit_transform(active_vectors)

assert not np.isnan(active_embedding).any(), (
    "UMAP produced NaN coordinates even after excluding all-zero rows -- inspect "
    "active_vectors for degenerate rows, or re-run with metric='euclidean'."
)

print(f"Built {expert_vectors.shape[0]} (layer, expert) vectors across {len(domains)} domains.")
print(f"UMAP embedding shape: {active_embedding.shape} ({int(active_mask.sum())} active of {expert_vectors.shape[0]} total pairs)")

TOP_K_TOKENS = 8
active_indices = np.flatnonzero(active_mask)

umap_points = []
for row, i in enumerate(active_indices):
    layer_id = int(point_layer_ids[i])
    expert_id = int(point_expert_ids[i])
    vec = expert_vectors[i]
    dominant_domain = domains[int(np.argmax(vec))]

    # top_tokens: pool every (token, routing score) pair that selected this (layer, expert)
    # across all domains, then keep the TOP_K_TOKENS with the highest score -- matching
    # extract_routing.ipynb's ranked-by-score approach, so the hover popup surfaces the
    # tokens that activated this expert most strongly, not just the first ones encountered.
    pooled = [
        (score, domain_tokens[d][t_idx], d)
        for d in domains
        for t_idx, score in expert_token_idx[d][layer_id][expert_id]
    ]
    pooled.sort(key=lambda item: -item[0])
    top_tokens = [
        {"token": tok, "score": round(float(score), 4), "domain": d}
        for score, tok, d in pooled[:TOP_K_TOKENS]
    ]

    umap_points.append({
        "layer_id": layer_id,
        "expert_id": expert_id,
        "x": round(float(active_embedding[row, 0]), 4),
        "y": round(float(active_embedding[row, 1]), 4),
        "dominant_domain": dominant_domain,
        "domain_activation_rate": {d: round(float(vec[j]), 4) for j, d in enumerate(domains)},
        "top_tokens": top_tokens,
    })

excluded_experts = [
    {"layer_id": int(point_layer_ids[i]), "expert_id": int(point_expert_ids[i])}
    for i in np.flatnonzero(~active_mask)
]

assert len(umap_points) + len(excluded_experts) == NUM_LAYERS * NUM_EXPERTS

umap_data = {
    "domains": domains,
    "num_layers": NUM_LAYERS,
    "num_experts": NUM_EXPERTS,
    "points": umap_points,
    "excluded_experts": excluded_experts,
}

with open(UMAP_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(umap_data, f, ensure_ascii=False, allow_nan=False, indent=2)

print(f"Wrote {UMAP_OUT_PATH} ({len(umap_points)} points, {len(excluded_experts)} excluded pairs)")


Built 1024 (layer, expert) vectors across 6 domains.
UMAP embedding shape: (1024, 2) (1024 active of 1024 total pairs)
Wrote domain_specialization_umap.json (1024 points, 0 excluded pairs)


In [9]:
try:
    from google.colab import files
    files.download(OUT_PATH)
    files.download(UMAP_OUT_PATH)
except ImportError:
    print("Not running in Google Colab -- skipping auto-download.")
    print(f"Files were written locally at: {OUT_PATH} and {UMAP_OUT_PATH}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>